# 05 — Schema versions одного датасета

Пишем последовательно в один target `default.stage_customers`, три раза подряд изменяя схему. Marquez для одного dataset'а (по `(namespace, name)`) будет хранить три **schema versions**.

Версии:
1. `(id, name, email)` — базовый набор.
2. `(id, name, email, country)` — добавлена колонка.
3. `(id, full_name, country)` — `name` переименована в `full_name`, `email` удалена.

Каждая ячейка вызывает `mode('overwrite')` → новая schema version.

> ⚙️ Под капотом Spark+Hive overwrite со сменой схемы — это **DROP+CREATE** Hive-таблицы (не Iceberg/Delta-style эволюция). Это нормально для демонстрации: Marquez аккумулирует `DatasetVersion` по `(namespace, name)` и история сохраняется даже при пересоздании.

Prerequisite: прогнан `00_setup.ipynb`. **Restart Jupyter kernel** перед запуском.

In [1]:
try:
    spark.stop()
except NameError:
    pass

from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .appName('lineage_05_schema_versions')
    .enableHiveSupport()
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
assert spark.sparkContext.appName == 'lineage_05_schema_versions', \
    f'appName fallback to {spark.sparkContext.appName!r} — restart Jupyter kernel'
print('app name:', spark.sparkContext.appName)

# Чистый старт перед первой версией.
spark.sql('DROP TABLE IF EXISTS default.stage_customers')

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/27 13:10:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/27 13:10:40 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.
26/05/27 13:10:53 WARN BlackbirdModule: Unable to find Java 9+ MethodHandles.privateLookupIn.  Blackbird is not performing optimally!
26/05/27 13:10:54 WARN SparkApplicationNameApplicationJobNameProvider: Failed to obtain the application name. Using the default value [unknown]. Set the [spark.app.name] or [spark.openlineage.appName] property.
26/05/27 13:10:54 WARN SparkApplicationNameApplicationJobNameProvider: Failed to obtain the application name. Using the default value [unknown]. Set the [spark.app.name] or [spark.openlineage.appName] property.


app name: lineage_05_schema_versions


26/05/27 13:10:59 WARN HiveConf: HiveConf of name hive.metastore.event.db.notification.api.auth does not exist
26/05/27 13:10:59 WARN HiveConf: HiveConf of name hive.log.dir does not exist
26/05/27 13:10:59 WARN HiveConf: HiveConf of name hive.root.logger does not exist
26/05/27 13:10:59 WARN HiveClientImpl: Detected HiveConf hive.execution.engine is 'tez' and will be reset to 'mr' to disable useless hive logic


DataFrame[]

## Версия 1: базовая схема (id, name, email)

In [2]:
df_v1 = spark.sql('SELECT id, name, email FROM default.raw_customers')
df_v1.write.mode('overwrite').saveAsTable('default.stage_customers')
spark.table('default.stage_customers').printSchema()

26/05/27 13:11:00 WARN InputFieldsCollector: Could not extract dataset identifier from org.apache.spark.sql.catalyst.analysis.ResolvedIdentifier
26/05/27 13:11:00 WARN InputFieldsCollector: Could not extract dataset identifier from org.apache.spark.sql.catalyst.analysis.ResolvedIdentifier
26/05/27 13:11:00 WARN OutputStatisticsOutputDatasetFacetBuilder: No jobId found in context
26/05/27 13:11:00 WARN InputFieldsCollector: Could not extract dataset identifier from org.apache.spark.sql.catalyst.analysis.ResolvedIdentifier
26/05/27 13:11:00 WARN InputFieldsCollector: Could not extract dataset identifier from org.apache.spark.sql.catalyst.analysis.ResolvedIdentifier
[Stage 0:=============================>                             (1 + 1) / 2]

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)



26/05/27 13:11:05 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


## Версия 2: добавлена колонка country

In [3]:
df_v2 = spark.sql('SELECT id, name, email, country FROM default.raw_customers')
df_v2.write.mode('overwrite').saveAsTable('default.stage_customers')
spark.table('default.stage_customers').printSchema()

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)



## Версия 3: переименование name→full_name, drop email

In [4]:
df_v3 = spark.sql('''
    SELECT
        id,
        name AS full_name,
        country
    FROM default.raw_customers
''')
df_v3.write.mode('overwrite').saveAsTable('default.stage_customers')
spark.table('default.stage_customers').printSchema()

[Stage 2:=============================>                             (1 + 1) / 2]

root
 |-- id: long (nullable = true)
 |-- full_name: string (nullable = true)
 |-- country: string (nullable = true)



In [5]:
spark.stop()

## Что смотреть в Marquez

1. UI → `default.stage_customers` → tab **Versions** — должно быть 3 записи.
2. API:
   ```bash
   curl 'http://localhost:5000/api/v1/namespaces/hadoop-cluster/datasets/default.stage_customers/versions' | jq '.versions[].fields | map(.name)'
   ```
   Должен вернуть 3 массива с разным составом полей. (`fields` — top-level в DatasetVersion-объекте; не `.schema.fields`.)
3. Column lineage для каждой версии свой: в v3 у `full_name` стрелка от `raw_customers.name` (даже под новым именем выходной колонки).